# 02 Plan A 3DGS

Goal:

1. prepare the Plan A workspace from `shared/images_512/` or a subset
2. run COLMAP feature extraction / matching / mapper
3. inspect the sparse reconstruction
4. organize the standard 3DGS input structure

This notebook only covers the COLMAP-based initialization pipeline.


In [19]:
from pathlib import Path
import shutil
import subprocess

CV_ROOT = Path('/home/bzhang512/CV_Project')
DATASET_ROOT = CV_ROOT / 'dataset'
SCENES = ['Re10k-1', 'DL3DV-2', '405841']
#SCENE = 'Re10k-1'   # 修改这里切换数据集
#SCENE = 'DL3DV-2'
SCENE = '405841'
MODE = 'colmap_96'   # 'colmap_full' or 'colmap_96'
#SUBSET_NAME = 'subset_96.txt'
SUBSET_NAME= 'subset_96_turnaware.txt' # 修改这里切换子集
OVERWRITE = False

SCENE_ROOT = DATASET_ROOT / SCENE
SHARED_ROOT = SCENE_ROOT / 'part1' / 'shared'
IMAGES_512_DIR = SHARED_ROOT / 'images_512'
SUBSETS_DIR = SHARED_ROOT / 'subsets'
PLAN_A_ROOT = SCENE_ROOT / 'part1' / 'planA'
WORK_DIR = PLAN_A_ROOT / MODE

print('WORK_DIR =', WORK_DIR)


WORK_DIR = /home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96


## 1. Dependency and environment check


In [20]:
def run(cmd, cwd=None):
    print("\n[CMD]", " ".join(cmd))
    return subprocess.run(cmd, cwd=cwd, check=True)

run(['which', 'colmap'])



[CMD] which colmap
/home/bzhang512/miniconda3/envs/3dgs/bin/colmap


CompletedProcess(args=['which', 'colmap'], returncode=0)

## 2. Prepare the COLMAP workspace


In [21]:
for d in [WORK_DIR, WORK_DIR / 'images', WORK_DIR / 'sparse', WORK_DIR / 'undistorted', WORK_DIR / 'gs_scene']:
    d.mkdir(parents=True, exist_ok=True)

if MODE == 'colmap_full':
    selected = sorted([p.name for p in IMAGES_512_DIR.iterdir() if p.is_file()])
else:
    subset_file = SUBSETS_DIR / SUBSET_NAME
    selected = [line.strip() for line in subset_file.read_text().splitlines() if line.strip()]

for name in selected:
    src = IMAGES_512_DIR / name
    dst = WORK_DIR / 'images' / name
    if dst.exists() and not OVERWRITE:
        continue
    shutil.copy2(src, dst)

print('prepared images:', len(selected))


prepared images: 96


## 3. Feature extraction


In [22]:
run([
    'colmap', 'feature_extractor',
    '--database_path', str(WORK_DIR / 'database.db'),
    '--image_path', str(WORK_DIR / 'images'),
    '--ImageReader.camera_model', 'SIMPLE_PINHOLE',
    '--ImageReader.single_camera', '1',
    '--SiftExtraction.max_num_features', '4096',
])



[CMD] colmap feature_extractor --database_path /home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/database.db --image_path /home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/images --ImageReader.camera_model SIMPLE_PINHOLE --ImageReader.single_camera 1 --SiftExtraction.max_num_features 4096


I20260327 03:49:28.654846 139765567549440 misc.cc:44] 
Feature extraction
I20260327 03:49:28.660580 139765097598976 sift.cc:720] Creating SIFT GPU feature extractor
I20260327 03:49:28.660830 139765089206272 sift.cc:720] Creating SIFT GPU feature extractor
I20260327 03:49:28.948148 139765080813568 feature_extraction.cc:260] Processed file [1/96]
I20260327 03:49:28.948222 139765080813568 feature_extraction.cc:263]   Name:            000082.png
I20260327 03:49:28.948227 139765080813568 feature_extraction.cc:272]   Dimensions:      512 x 512
I20260327 03:49:28.948231 139765080813568 feature_extraction.cc:275]   Camera:          #1 - SIMPLE_PINHOLE
I20260327 03:49:28.948237 139765080813568 feature_extraction.cc:278]   Focal Length:    614.40px
I20260327 03:49:28.948249 139765080813568 feature_extraction.cc:282]   Features:        1865 (SIFT)
I20260327 03:49:29.053257 139765080813568 feature_extraction.cc:260] Processed file [2/96]
I20260327 03:49:29.053303 139765080813568 feature_extraction

CompletedProcess(args=['colmap', 'feature_extractor', '--database_path', '/home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/database.db', '--image_path', '/home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/images', '--ImageReader.camera_model', 'SIMPLE_PINHOLE', '--ImageReader.single_camera', '1', '--SiftExtraction.max_num_features', '4096'], returncode=0)

## 4. Sequential matching


In [23]:
run([
    'colmap', 'sequential_matcher',
    '--database_path', str(WORK_DIR / 'database.db'),
    '--SequentialMatching.overlap', '10',
    '--SequentialMatching.quadratic_overlap', '1',
])



[CMD] colmap sequential_matcher --database_path /home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/database.db --SequentialMatching.overlap 10 --SequentialMatching.quadratic_overlap 1


I20260327 03:49:31.655232 139851110010880 misc.cc:44] 
Feature matching & geometric verification
I20260327 03:49:31.890257 139851101618176 feature_matching_utils.cc:78] Bind FeatureMatcherWorker to GPU device 0
I20260327 03:49:31.890788 139851101618176 sift.cc:1446] Creating SIFT GPU feature matcher
I20260327 03:49:31.901650 139851093225472 feature_matching_utils.cc:78] Bind FeatureMatcherWorker to GPU device 1
I20260327 03:49:31.901686 139851093225472 sift.cc:1446] Creating SIFT GPU feature matcher
I20260327 03:49:31.902157 139851110010880 pairing.cc:412] Generating sequential image pairs...
I20260327 03:49:31.902682 139851110010880 pairing.cc:470] Processing image [1/96]
I20260327 03:49:31.914861 139851110010880 feature_matching.cc:117] in 0.012s
I20260327 03:49:31.914920 139851110010880 pairing.cc:470] Processing image [2/96]
I20260327 03:49:31.923152 139851110010880 feature_matching.cc:117] in 0.008s
I20260327 03:49:31.923175 139851110010880 pairing.cc:470] Processing image [3/96]


CompletedProcess(args=['colmap', 'sequential_matcher', '--database_path', '/home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/database.db', '--SequentialMatching.overlap', '10', '--SequentialMatching.quadratic_overlap', '1'], returncode=0)

## 5. Mapper


In [24]:
run([
    'colmap', 'mapper',
    '--database_path', str(WORK_DIR / 'database.db'),
    '--image_path', str(WORK_DIR / 'images'),
    '--output_path', str(WORK_DIR / 'sparse'),
])



[CMD] colmap mapper --database_path /home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/database.db --image_path /home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/images --output_path /home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/sparse


I20260327 03:49:34.329805 139720316796928 incremental_pipeline.cc:264] Loading database
I20260327 03:49:34.333051 139720316796928 database_cache.cc:67] Loading rigs...
I20260327 03:49:34.333200 139720316796928 database_cache.cc:77]  1 in 0.000s
I20260327 03:49:34.333251 139720316796928 database_cache.cc:85] Loading cameras...
I20260327 03:49:34.333294 139720316796928 database_cache.cc:103]  1 in 0.000s
I20260327 03:49:34.333330 139720316796928 database_cache.cc:111] Loading frames...
I20260327 03:49:34.333465 139720316796928 database_cache.cc:126]  96 in 0.000s
I20260327 03:49:34.333502 139720316796928 database_cache.cc:134] Loading matches...
I20260327 03:49:34.339525 139720316796928 database_cache.cc:139]  545 in 0.006s
I20260327 03:49:34.339593 139720316796928 database_cache.cc:155] Loading images...
I20260327 03:49:34.346893 139720316796928 database_cache.cc:239]  96 in 0.007s (connected 96)
I20260327 03:49:34.346954 139720316796928 database_cache.cc:250] Building correspondence gr

CompletedProcess(args=['colmap', 'mapper', '--database_path', '/home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/database.db', '--image_path', '/home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/images', '--output_path', '/home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/sparse'], returncode=0)

## 6. Model inspection


In [25]:
sparse_root = WORK_DIR / 'sparse'
model_dirs = sorted([p for p in sparse_root.iterdir() if p.is_dir()]) if sparse_root.exists() else []
print('detected sparse models:')
for p in model_dirs:
    print(' -', p)

for p in model_dirs:
    print("\n[analyze]", p)
    run(['colmap', 'model_analyzer', '--path', str(p)])

print("\nDefault downstream path is still:", sparse_root / "0")



detected sparse models:
 - /home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/sparse/0

[analyze] /home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/sparse/0

[CMD] colmap model_analyzer --path /home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/sparse/0

Default downstream path is still: /home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/sparse/0


I20260327 03:50:23.106040 140294024249344 model.cc:451] Rigs: 1
I20260327 03:50:23.106428 140294024249344 model.cc:452] Cameras: 1
I20260327 03:50:23.106437 140294024249344 model.cc:453] Frames: 96
I20260327 03:50:23.106440 140294024249344 model.cc:454] Registered frames: 96
I20260327 03:50:23.106443 140294024249344 model.cc:456] Images: 96
I20260327 03:50:23.106445 140294024249344 model.cc:457] Registered images: 96
I20260327 03:50:23.106455 140294024249344 model.cc:459] Points: 4991
I20260327 03:50:23.106458 140294024249344 model.cc:460] Observations: 58831
I20260327 03:50:23.106465 140294024249344 model.cc:462] Mean track length: 11.787417
I20260327 03:50:23.106486 140294024249344 model.cc:464] Mean observations per image: 612.822917
I20260327 03:50:23.106493 140294024249344 model.cc:467] Mean reprojection error: 0.467356px


In [26]:
# Optional direct check for the default chosen model
run(['colmap', 'model_analyzer', '--path', str(WORK_DIR / 'sparse' / '0')])



[CMD] colmap model_analyzer --path /home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/sparse/0


I20260327 03:50:29.288426 140129191870464 model.cc:451] Rigs: 1
I20260327 03:50:29.288814 140129191870464 model.cc:452] Cameras: 1
I20260327 03:50:29.288827 140129191870464 model.cc:453] Frames: 96
I20260327 03:50:29.288830 140129191870464 model.cc:454] Registered frames: 96
I20260327 03:50:29.288833 140129191870464 model.cc:456] Images: 96
I20260327 03:50:29.288835 140129191870464 model.cc:457] Registered images: 96
I20260327 03:50:29.288844 140129191870464 model.cc:459] Points: 4991
I20260327 03:50:29.288846 140129191870464 model.cc:460] Observations: 58831
I20260327 03:50:29.288853 140129191870464 model.cc:462] Mean track length: 11.787417
I20260327 03:50:29.288871 140129191870464 model.cc:464] Mean observations per image: 612.822917
I20260327 03:50:29.288878 140129191870464 model.cc:467] Mean reprojection error: 0.467356px


CompletedProcess(args=['colmap', 'model_analyzer', '--path', '/home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/sparse/0'], returncode=0)

## 7. Undistort and organize gs_scene


In [27]:
run([
    'colmap', 'image_undistorter',
    '--image_path', str(WORK_DIR / 'images'),
    '--input_path', str(WORK_DIR / 'sparse' / '0'),
    '--output_path', str(WORK_DIR / 'gs_scene'),
    '--output_type', 'COLMAP',
])

print('Plan A gs_scene ready at', WORK_DIR / 'gs_scene')



[CMD] colmap image_undistorter --image_path /home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/images --input_path /home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/sparse/0 --output_path /home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/gs_scene --output_type COLMAP
Plan A gs_scene ready at /home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/gs_scene


I20260327 03:50:31.038467 139766627868672 misc.cc:44] 
Reading reconstruction
I20260327 03:50:31.060384 139766627868672 image.cc:374] => Reconstruction with 96 images and 4991 points
I20260327 03:50:31.060505 139766627868672 misc.cc:44] 
Image undistortion
I20260327 03:50:31.066292 139766627868672 undistortion.cc:197] Undistorting image [1/96]
I20260327 03:50:31.066430 139763213307904 undistortion.cc:239] Undistorted image found; copying to location: /home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/gs_scene/images/000082.png
I20260327 03:50:31.068386 139766627868672 undistortion.cc:197] Undistorting image [2/96]
I20260327 03:50:31.068514 139763213307904 undistortion.cc:239] Undistorted image found; copying to location: /home/bzhang512/CV_Project/dataset/405841/part1/planA/colmap_96/gs_scene/images/000081.png
I20260327 03:50:31.070130 139766627868672 undistortion.cc:197] Undistorting image [3/96]
I20260327 03:50:31.070377 139763213307904 undistortion.cc:239] Undistorted i